# Definitive fair-control check — tunnel topology defect (scan105)

Self-contained, single-cell diagnostic. Loads everything fresh from disk
(project root, latest `scan105-*` run checkpoint, cached surface samples),
so it does **not** depend on any state from `pipeline_run_all_fix_3` or
`roadmap_execution`. Just run this notebook in the same environment that
has access to `SDF-GS-Refine-implementation/` and `run_artifacts/scan105-*/`.

**What it fixes vs. the earlier (invalid) check:** the previous control
point was picked as "the Gaussian farthest from the tunnel," which turned
out to sit in empty space (0 surface samples within radius, nearest sample
0.285 units away) — not a fair comparison. This version instead picks the
control point *from the cached surface samples themselves*, matched to the
tunnel region's local point density, so the eikonal and normal-consistency
stats are apples-to-apples.

Run once. The verdict at the end is final — it tells you which stage to
fix next (SDF/training-loss side vs. extraction/post-processing side).

In [1]:
# ==================================================================
# DEFINITIVE CHECK: fair-control eikonal + normal-consistency test.
# Fixes the flaw in the previous control point (it was the Gaussian
# farthest from the tunnel, which happened to have ZERO surface
# samples nearby -- an empty-space point, not a fair comparison).
# This cell instead picks a control point FROM the cached surface
# samples themselves, matched on local point density to the tunnel
# region, so eikonal/normal stats are apples-to-apples.
# ==================================================================
import sys
from pathlib import Path
import numpy as np
import torch
from scipy.spatial import cKDTree

# ---- 0. Locate project + most recent scan105 run ----
def _find_project_root(marker="src/utils.py"):
    for root in (Path("/notebooks"), Path("/content"), Path("/kaggle/working"), Path.home(), Path.cwd()):
        if root.exists():
            for c in root.glob("**/SDF-GS-Refine-implementation"):
                if (c / marker).exists():
                    return c
    return None

PROJECT_ROOT = _find_project_root()
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate SDF-GS-Refine-implementation project root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import sdf as sdf_mod
from src.utils import load_config

CFG = load_config(str(PROJECT_ROOT / "configs" / "config.json"))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

run_dir = sorted(PROJECT_ROOT.glob("run_artifacts/scan105-*/02_training_checkpoint.pt"),
                  key=lambda p: p.stat().st_mtime, reverse=True)[0].parent
print(f"[OK] Run dir: {run_dir}")

ckpt = _torch_load(run_dir / "02_training_checkpoint.pt")
surf_ckpt = _torch_load(sorted(run_dir.glob("*surface*sample*.pt"))[0])
surf_pts = surf_ckpt["surface_points"].cpu().numpy() if torch.is_tensor(surf_ckpt["surface_points"]) else np.asarray(surf_ckpt["surface_points"])
surf_nrm = surf_ckpt["surface_normals"].cpu().numpy() if torch.is_tensor(surf_ckpt["surface_normals"]) else np.asarray(surf_ckpt["surface_normals"])
tree = cKDTree(surf_pts)

TUNNEL_CENTROID = np.array([0.1353, -0.1619, -0.0745])
RADIUS = 0.03

# ---- 1. Pick a FAIR control point: on-surface, density-matched ----
tunnel_idx = tree.query_ball_point(TUNNEL_CENTROID, RADIUS)
tunnel_density = len(tunnel_idx)
print(f"[TUNNEL] {tunnel_density} surface points within r={RADIUS}")

rng = np.random.default_rng(0)
candidate_idx = rng.choice(len(surf_pts), size=min(5000, len(surf_pts)), replace=False)
best_pt, best_score, best_n = None, -1, None
for i in candidate_idx:
    c = surf_pts[i]
    d_to_tunnel = np.linalg.norm(c - TUNNEL_CENTROID)
    if d_to_tunnel < RADIUS * 4:  # must be genuinely away from tunnel
        continue
    n_local = len(tree.query_ball_point(c, RADIUS))
    # score: prefer density close to tunnel's, and far from tunnel
    density_match = -abs(n_local - tunnel_density)
    if density_match > best_score and n_local > 4:
        best_score, best_pt, best_n = density_match, c, n_local

if best_pt is None:
    raise RuntimeError("Could not find a density-matched control point on the surface.")
print(f"[CONTROL] chosen at {np.round(best_pt,4)}, dist_to_tunnel={np.linalg.norm(best_pt-TUNNEL_CENTROID):.3f}, "
      f"{best_n} surface points within r={RADIUS} (tunnel had {tunnel_density})")

# ---- 2. Eikonal check: tunnel vs fair control ----
sdf_cfg = CFG["sdf_network"]
sdf_net = sdf_mod.SDFNetwork(d_in=3, d_hidden=sdf_cfg["d_hidden"], n_layers=sdf_cfg["n_layers"],
                              radius_init=sdf_cfg["radius_init"], beta=sdf_cfg["softplus_beta"]).to(DEVICE)
sdf_net.load_state_dict(ckpt["sdf_state"])
sdf_net.eval()

def _ball_sample(center, radius, n, rng):
    d = rng.normal(size=(n, 3)); d /= np.linalg.norm(d, axis=1, keepdims=True)
    r = radius * (rng.uniform(size=(n, 1)) ** (1/3))
    return center[None, :] + d * r

n_probe = 2000
tunnel_probe = torch.as_tensor(_ball_sample(TUNNEL_CENTROID, RADIUS, n_probe, rng), dtype=torch.float32, device=DEVICE)
control_probe = torch.as_tensor(_ball_sample(best_pt, RADIUS, n_probe, rng), dtype=torch.float32, device=DEVICE)

_, grad_t = sdf_net.gradient(tunnel_probe)
_, grad_c = sdf_net.gradient(control_probe)
eik_t = (grad_t.norm(dim=-1) - 1.0).abs().detach()
eik_c = (grad_c.norm(dim=-1) - 1.0).abs().detach()
ratio = float(eik_t.mean() / max(eik_c.mean().item(), 1e-8))
print(f"\n[EIKONAL] tunnel mean dev={eik_t.mean():.4f} | fair-control mean dev={eik_c.mean():.4f} | ratio={ratio:.2f}x")

# ---- 3. Normal-consistency: tunnel vs fair control ----
def _normal_report(center, radius, label):
    idx = tree.query_ball_point(center, radius)
    nrm = surf_nrm[idx]; nrm /= (np.linalg.norm(nrm, axis=1, keepdims=True) + 1e-12)
    mean_dir = nrm.mean(axis=0); mean_dir /= (np.linalg.norm(mean_dir) + 1e-12)
    frac_opposing = float((nrm @ mean_dir < 0).mean())
    print(f"[{label}] n={len(idx)}  frac_opposing={frac_opposing:.1%}")
    return frac_opposing

frac_t = _normal_report(TUNNEL_CENTROID, RADIUS, "TUNNEL")
frac_c = _normal_report(best_pt, RADIUS, "CONTROL")

# ---- 4. ONE definitive verdict ----
print("\n=== DEFINITIVE VERDICT ===")
eik_flag = ratio > 2.0
normal_flag = frac_t > 0.15 and frac_t > 3 * max(frac_c, 0.01)
if eik_flag and normal_flag:
    print(">>> CONFIRMED: genuine close-conflicting-sheets geometric ambiguity at the tunnel, "
          "verified against a fair on-surface control. This is the root cause. "
          "Proceed straight to the fix: retrain with normal_consistency_penalty and/or "
          "eikonal_loss_weighted boosted at the flagged region -- no further hypothesis testing needed.")
elif eik_flag and not normal_flag:
    print(">>> Eikonal instability is real and localized, but NOT explained by conflicting normals. "
          "Root cause is an under-constrained SDF region, not close sheets. Fix: boost eikonal "
          "sample weight/density specifically here (eikonal_loss_weighted), not normal-consistency loss.")
elif normal_flag and not eik_flag:
    print(">>> CONFIRMED (topology-only variant): conflicting normals are real and well above the fair "
          "control baseline, but the eikonal field stays healthy in this region -- so this is NOT a "
          "numerically under-constrained SDF. Two nearby, gradient-stable surface sheets face opposite "
          "directions closely enough that marching cubes stitches them into a spurious handle/tunnel. "
          "Fix: retrain with normal_consistency_penalty targeted at this region (eikonal_loss_weighted "
          "will not help, since eikonal is already fine here). Worth also spot-checking Stage D/E "
          "(marching cubes threshold, grid resolution) in parallel, since clean-eikonal-but-wrong-topology "
          "can sometimes be an extraction artifact rather than a training one.")
else:
    print(">>> NOT CONFIRMED against a fair control: the tunnel does not show meaningfully worse "
          "eikonal or normal behavior than a comparable on-surface region. The topology defect may be "
          "coming from the extraction/post-processing stage (marching cubes threshold, grid resolution) "
          "rather than the SDF/Gaussian fitting stage -- check Stage D/E next, not the network.")


[OK] Run dir: /notebooks/Generated/SDF-GS-Refine-implementation/run_artifacts/scan105-498cee738663c8be
[TUNNEL] 331 surface points within r=0.03
[CONTROL] chosen at [ 0.1623 -0.3157 -0.0963], dist_to_tunnel=0.158, 331 surface points within r=0.03 (tunnel had 331)

[EIKONAL] tunnel mean dev=0.3368 | fair-control mean dev=0.6046 | ratio=0.56x
[TUNNEL] n=331  frac_opposing=22.4%
[CONTROL] n=331  frac_opposing=5.1%

=== DEFINITIVE VERDICT ===
>>> CONFIRMED (topology-only variant): conflicting normals are real and well above the fair control baseline, but the eikonal field stays healthy in this region -- so this is NOT a numerically under-constrained SDF. Two nearby, gradient-stable surface sheets face opposite directions closely enough that marching cubes stitches them into a spurious handle/tunnel. Fix: retrain with normal_consistency_penalty targeted at this region (eikonal_loss_weighted will not help, since eikonal is already fine here). Worth also spot-checking Stage D/E (marching cube